In [ ]:
!pip install datasets

In [ ]:
import os
import json
from datasets import load_dataset
from PIL import Image
from tqdm import tqdm

TRAIN_DIR = "/kaggle/working/plan_b_train_data"
TARGET_SIZE = 1024
os.makedirs(TRAIN_DIR, exist_ok=True)

# 1. Download ONLY the first 50 images for quick testing
print("Downloading FIRST 50 images from pixel-art-nouns-2k for test run...")
dataset = load_dataset("jiovine/pixel-art-nouns-2k", split="train[:50]")   # ← ONLY CHANGE HERE

metadata = []
print(f"Upscaling {len(dataset)} images using Nearest Neighbor...")

for i, item in enumerate(tqdm(dataset)):
    try:
        img = item['image']
        
        # Keep your style trigger (good for pixel art)
        caption = "pixel art style, " + item['text'] 
        
        if img.mode != "RGB":
            img = img.convert("RGB")
            
        # Nearest Neighbor upscaling
        img_upscaled = img.resize((TARGET_SIZE, TARGET_SIZE), resample=Image.NEAREST)
        
        filename = f"noun_{i}.png"
        img_upscaled.save(os.path.join(TRAIN_DIR, filename), "PNG")
        
        metadata.append({"file_name": filename, "text": caption})
        
    except Exception as e:
        print(f"Skipping image {i} due to error: {e}")

# 3. Save the JSONL file for the trainer
with open(os.path.join(TRAIN_DIR, "metadata.jsonl"), 'w') as f:
    for entry in metadata:
        f.write(json.dumps(entry) + "\n")
        
print(f"✅ Test data ready! Only {len(dataset)} images saved to {TRAIN_DIR}")

In [ ]:
# import os
# import json
# from datasets import load_dataset
# from PIL import Image
# from tqdm import tqdm

# TRAIN_DIR = "/kaggle/working/plan_b_train_data"
# TARGET_SIZE = 1024  # Upscale target for SSD-1B
# os.makedirs(TRAIN_DIR, exist_ok=True)

# # 1. Download directly from Hugging Face
# print("Downloading pixel-art-nouns-2k from Hugging Face...")
# dataset = load_dataset("jiovine/pixel-art-nouns-2k", split="train")

# metadata = []
# print(f"Upscaling {len(dataset)} images using Nearest Neighbor...")

# for i, item in enumerate(tqdm(dataset)):
#     try:
# #!pip install datasets        # HF datasets automatically load images as PIL objects
#         img = item['image']
        
#         # The text column in this specific dataset is usually 'text'
#         # Add the mandatory style trigger here to prevent catastrophic forgetting
#         caption = "pixel art style, " + item['text'] 
        
#         if img.mode != "RGB":
#             img = img.convert("RGB")
            
#         # 2. Crucial Step: Nearest Neighbor upscaling to keep pixels sharp
#         img_upscaled = img.resize((TARGET_SIZE, TARGET_SIZE), resample=Image.NEAREST)
        
#         filename = f"noun_{i}.png"
#         img_upscaled.save(os.path.join(TRAIN_DIR, filename), "PNG")
        
#         metadata.append({"file_name": filename, "text": caption})
        
#     except Exception as e:
#         print(f"Skipping image {i} due to error: {e}")

# # 3. Save the JSONL file for the trainer
# with open(os.path.join(TRAIN_DIR, "metadata.jsonl"), 'w') as f:
#     for entry in metadata:
#         f.write(json.dumps(entry) + "\n")
        
# print(f"Plan B Data Ready! Saved to {TRAIN_DIR}")

In [ ]:
# === PIXART-SIGMA PIPELINE SETUP (STABLE FOR T4 x2) ===
!pip install -U -q diffusers accelerate transformers peft bitsandbytes wandb

# Use the stable PixArt-alpha repo (same Sigma model, reliable LoRA saving)
!git clone https://github.com/PixArt-alpha/PixArt-alpha.git
%cd PixArt-alpha

# Install requirements
!pip install -r requirements.txt

print("✅ Stable PixArt-Sigma LoRA pipeline ready for T4 x2!")

In [ ]:
# # ============================================================
# # W&B Setup — load API key from Kaggle Secrets and login
# # Make sure you've added your W&B API key as a Kaggle Secret
# # with the label "WANDB" (Add-ons → Secrets in notebook sidebar)
# # ============================================================
# import os
# import wandb
# from kaggle_secrets import UserSecretsClient

# wandb_api_key = UserSecretsClient().get_secret("WANDB")
# wandb.login(key=wandb_api_key)

# # Set the W&B project name so the training script picks it up
# os.environ["WANDB_PROJECT"] = "Pixel Art"

# print("Logged into W&B successfully. Project: 'Pixel Art'")

In [ ]:
import os
from accelerate.utils import write_basic_config

write_basic_config()

TRAIN_DIR = "/kaggle/working/plan_b_train_data"
MODEL_ID = "PixArt-alpha/PixArt-Sigma-XL-2-1024-MS"
OUTPUT_DIR = "/kaggle/working/pixart_sigma_pixelart_lora_TEST"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# === QUICK TEST RUN — optimized for T4 x2 ===
!accelerate launch --num_processes=2 --mixed_precision="fp16" \
    train_scripts/train_pixart_lora_hf.py \
    --pretrained_model_name_or_path=$MODEL_ID \
    --train_data_dir=$TRAIN_DIR \
    --caption_column="text" \
    --resolution=1024 \
    --train_batch_size=2 \
    --gradient_accumulation_steps=4 \
    --max_train_steps=50 \
    --learning_rate=1e-5 \
    --lr_scheduler="cosine" \
    --lr_warmup_steps=50 \
    --output_dir=$OUTPUT_DIR \
    --checkpointing_steps=10 \
    --validation_prompt="a cute pixel art cat wearing sunglasses" \
    --validation_epochs=5 \
    # --report_to="wandb" \
    --mixed_precision="fp16" \
    --max_token_length=300 \
    --use_8bit_adam \
    --gradient_checkpointing \
    --noise_offset=0.1 \
    --rank=16 \
    --lora_alpha=16

print(f"✅ Quick test finished! LoRA saved to {OUTPUT_DIR}")

In [ ]:
import torch
from diffusers import Transformer2DModel, PixArtSigmaPipeline
from peft import PeftModel
import os
from PIL import Image

OUTPUT_DIR = "/kaggle/working/pixart_sigma_pixelart_lora_TEST"

# === DEBUG: Show exactly what was saved ===
print("📁 Files in OUTPUT_DIR:")
for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(OUTPUT_DIR, '').count(os.sep)
    indent = ' ' * 4 * level
    print(f"{indent}📂 {os.path.basename(root)}/")
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        print(f"{subindent}📄 {f}")

# === Find the correct LoRA path (most common locations) ===
lora_path = None
for root, dirs, files in os.walk(OUTPUT_DIR):
    if "adapter_config.json" in files and "pytorch_lora_weights.safetensors" in files:
        lora_path = root
        print(f"\n✅ Found LoRA files in: {lora_path}")
        break

if lora_path is None:
    # Fallback - some scripts save only in the root
    if os.path.exists(os.path.join(OUTPUT_DIR, "pytorch_lora_weights.safetensors")):
        lora_path = OUTPUT_DIR
        print(f"\n✅ Found LoRA weights in main folder: {OUTPUT_DIR}")
    else:
        raise FileNotFoundError("❌ Could not find any LoRA files. Training may have failed to save them.")

# Load base model
transformer = Transformer2DModel.from_pretrained(
    "PixArt-alpha/PixArt-Sigma-XL-2-1024-MS",
    subfolder="transformer",
    torch_dtype=torch.float16,
)

# Load your trained LoRA
transformer = PeftModel.from_pretrained(transformer, lora_path)

# Create pipeline
pipe = PixArtSigmaPipeline.from_pretrained(
    "PixArt-alpha/pixart_sigma_sdxlvae_T5_diffusers",
    transformer=transformer,
    torch_dtype=torch.float16,
)
pipe.to("cuda")
pipe.enable_model_cpu_offload()

# Generate test image
image = pipe(
    prompt="a pixel art character with square black glasses and a hotdog-shaped head",
    negative_prompt="blur, low quality, realistic, photo",
    num_inference_steps=50,
    guidance_scale=4.5,
    height=1024,
    width=1024,
).images[0]

# Downscale to 256×256
image = image.resize((256, 256), resample=Image.NEAREST)
image.save("pixel_art_test.png")
image

In [ ]:
# import os

# OUTPUT_DIR = "/kaggle/working/pixart_sigma_pixelart_lora_TEST"

# print("=== FULL DIRECTORY CONTENTS OF OUTPUT_DIR ===")
# for root, dirs, files in os.walk(OUTPUT_DIR):
#     level = root.replace(OUTPUT_DIR, '').count(os.sep)
#     indent = '    ' * level
#     print(f"{indent}📂 {os.path.basename(root)}/")
#     subindent = '    ' * (level + 1)
#     for f in sorted(files):
#         print(f"{subindent}📄 {f}")

# print("\n✅ Directory listing complete. Copy-paste the entire output here.")